# <center> Кластеризация изображений транспортных средств

## Постановка задачи

<center> <img src=https://i.ibb.co/t8DvkyB/smart-city-image-1.jpg align="right" width="300"/> </center>
<center> <img src=https://i.ibb.co/qYkWNVh/smart-city-image-3.jpg align="right" width="300"/> </center>


Один из ключевых проектов IntelliVision — Smart City/Transportation, система, обеспечивающая безопасность дорожного движения и более эффективную работу парковок. С помощью Smart City/Transportation можно контролировать сигналы светофоров и соблюдение ограничений скорости, определять виды транспортных средств, распознавать номерные знаки, считать автомобили и людей.

В основе всех перечисленных возможностей проекта лежит CV (Computer Vision, компьютерное зрение). Чтобы их реализовать, компания использует модели, для обучения которых применяются огромные размеченные датасеты с изображениями транспортных средств. Однако система работает в режиме реального времени и с каждым днём данных становится всё больше. Алгоритм нуждается в постоянной модернизации и должен учитывать множество факторов.

Для модификации и повышения эффективности системы Smart City/Transportation команде необходимо автоматизировать определение дополнительных параметров авто на изображении:

* тип автомобиля (кузова),
* ракурс снимка (вид сзади/спереди),
* цвет автомобиля,
* другие характеристики.

Также необходимо автоматизировать поиск выбросов в данных (засветы и блики на изображениях, изображения, на которых отсутствуют автомобили и т. д.).

К сожалению, у компании нет комплексной модели, которая могла бы одновременно находить на изображении автомобиль и определять все нужные параметры. Её нужно построить, однако многокомпонентная разметка новых данных по всем этим параметрам — очень трудозатратное занятие, которое стоит больших денег.

При решении задачи разметки данных у команды возникла гипотеза, которая нуждается в исследовании.


**Гипотеза:** разметку исходных данных можно эффективно провести с помощью методов кластеризации. 


**В чём идея?**

*Давайте будем использовать небольшой набор моделей свёрточных нейронных сетей, обученных на различных датасетах и решающих различные задачи от классификации изображений по цвету до классификации типов транспортных средств, пропустим нашу базу изображений через каждую модель, но возьмём не выходной результат модели, а только промежуточное представление признаков (дескриптор), полученное на свёрточных слоях сети.*

*Выполним такую операцию для всех изображений из набора данных, на основе полученных дескрипторов кластеризуем изображения, проинтерпретируем полученные кластеры и попробуем найти в них необходимую информацию.*

Теперь, когда мы обсудили гипотезу, перейдём к постановке задачи.

<center> <img src=https://i.ibb.co/hLcBpZF/2023-03-27-12-11-17.png align="right" width="500"/> </center>

У нас будет набор из 416 314 изображений транспортных средств различных типов, цветов и снятых с разных ракурсов.

Команда IntelliVision уже обработала свой набор данных с помощью нескольких моделей глубокого обучения (свёрточных нейронных сетей) и получила четыре варианта вектора признаков (дескрипторов) для каждого изображения.

**Наша задача** — используя готовые дескрипторы, разбить изображения на кластеры и проинтерпретировать каждый из них. Для всех вариантов дескрипторов нужно применить несколько алгоритмов кластеризации и сравнить полученные результаты. Сравнивать можно на основе метрик, визуализаций плотностей кластеров и по тому, насколько хорошо интерпретируются кластеры.

Дополнительная подзадача — найти выбросы среди изображений. Это могут быть изображения плохого качества, изображения с бликами или изображения, на которых нет транспортных средств и т. д.

Бизнес-задача: исследовать возможность применения алгоритмов кластеризации для разметки новых данных и поиска выбросов.

Техническая задача для нас как для специалиста в Data Science: построить модель кластеризации изображений на основе дескрипторов, выделяемых с помощью различных архитектур нейронных сетей, проинтерпретировать полученные результаты и выбрать модель или комбинацию моделей, которая выделяет наиболее пригодные для интерпретации признаки.

**Наши основные цели:**
1. Для каждого типа дескрипторов необходимо:
    * выполнить предобработку дескрипторов;
    * произвести кластеризацию изображений на основе их дескрипторов, подобрав алгоритм и параметры кластеризации;
    * сделать визуализацию полученных кластеров в 2D- или 3D-пространстве;
    * проинтерпретировать полученные кластеры — в паре предложений сформулировать, какие изображения попали в каждый из кластеров.
2. Сравнить между собой полученные кластеризации для каждого типа дескрипторов (по метрикам, визуализации и результатам интерпретации).
3. Выполнить автоматизированный поиск выбросов среди изображений на основе дескрипторов.
4. Дополнительная задача: попробовать воспользоваться смесью дескрипторов, полученных различными моделями, и проинтерпретировать полученные результаты.

**Примечание.** При выборе алгоритма кластеризации будем ориентироваться на внутренние метрики, а именно на индекс Калински — Харабаса (`calinski_harabasz_score`) и индекс Дэвиса — Болдина (`davies_bouldin_score`), а также на интерпретируемость кластеров и визуализацию.

## Данные и их описание

Исходная папка с данными имеет следующую структуру:

```
IntelliVision_case
├─descriptors
    └─efficientnet-b7.pickle
    └─osnet.pickle
    └─vdc_color.pickle
    └─vdc_type.pickle
├─row_data
    └─veriwild.zip
├─images_paths.csv 
```

Давайте разберёмся в ней:

* В папке `descriptors` содержатся дескрипторы, полученные для каждого из изображений с помощью соответствующих нейронных сетей, в формате numpy-массивов, сохранённых в файлах pickle:
    * `efficientnet-b7.pickle` — дескрипторы, выделенные моделью классификации с архитектурой EfficientNet версии 7. Эта модель является свёрточной нейронной сетью, предобученной на на датасете ImageNet, в котором содержатся изображения более 1000 различных классов. Эта модель при обучении не видела датасета veriwiId. 

    * `osnet.pickle` — дескрипторы, выделенные моделью OSNet, обученной для детектирования людей, животных и машин. Модель не обучалась на исходном датасете veriwiId.

    * `vdc_color.pickle` — дескрипторы, выделенные моделью регрессии для определения цвета транспортных средств в формате RGB. Частично обучена на исходном датасете veriwild.
    
    * `vdc_type.pickle` — дескрипторы, выделенные моделью классификации транспортных средств по типу на десяти классах. Частично обучена на исходном датасете veriwild.

* В папке `row_data` содержится zip-архив с исходными изображениями автомобилей. Распакуйте его содержимое в папку row_data. Архив содержит десять папок с изображениями, пронумерованных от 1 до 10. Каждая папка содержит подпапки, обозначенные пятизначными цифрами, например 36191. 

В каждой из таких подпапок содержатся фотографии одного конкретного автомобиля с разных ракурсов, снятые с помощью дорожных видеокамер.

* В файле `images_paths.csv` представлен список из полных путей до изображений. Он пригодится вам при анализе изображений, попавших в определённый кластер.


Импорт базовых библиотек:

In [ ]:
import pickle
import pandas as pd
import numpy as np

import warnings 

from IPython.display import display, HTML

warnings.filterwarnings("ignore")

plt.rcParams["patch.force_edgecolor"] = True

## 1. Знакомство со структурой данных

Прочитаем numpy-массивы из предоставленных pickle-файлов.

Примечание Для удобства дальнейшей работы составим четыре DataFrame с путями до изображений и соответствующими им дескрипторами.

Посмотрим на размерности каждой из четырёх заданных матриц и сравните использованные модели глубокого обучения по размерностям выходных дескрипторов изображений.

In [ ]:
# Базовый путь к датасету
BASE_PATH = '/kaggle/input/datasets/markhomeless/intellivision-case/IntelliVision_case'

# Пути к дескрипторам
efficientnet_path = f'{BASE_PATH}/descriptors/efficientnet-b7.pickle'
osnet_path = f'{BASE_PATH}/descriptors/osnet.pickle'
vdc_color_path = f'{BASE_PATH}/descriptors/vdc_color.pickle'
vdc_type_path = f'{BASE_PATH}/descriptors/vdc_type.pickle'

In [ ]:
import os

# Загружаем пути к изображениям (уже сделали)
paths_df = pd.read_csv(images_paths)
print(f"Загружено {len(paths_df)} путей к изображениям")

# Функция для загрузки дескрипторов
def load_descriptors(file_path, name):
    print(f"\nЗагрузка {name}...")
    if os.path.exists(file_path):
        with open(file_path, 'rb') as f:
            descriptors = pickle.load(f)
        print(f"  Файл: {file_path}")
        print(f"  Форма массива: {descriptors.shape}")
        print(f"  Тип данных: {descriptors.dtype}")
        return descriptors
    else:
        print(f"  Файл НЕ НАЙДЕН: {file_path}")
        return None

# Загружаем все дескрипторы
X_ef = load_descriptors(efficientnet_path, "EfficientNet")
X_osnet = load_descriptors(osnet_path, "OSNet")
X_color = load_descriptors(vdc_color_path, "VDC Color")
X_type = load_descriptors(vdc_type_path, "VDC Type")

## 2. Преобразование, очистка и анализ данных

Признаки, найденные с помощью некоторых моделей, исчисляются тысячами, что довольно много, учитывая общее количество наблюдений.

Как вы понимаете, производить кластеризацию на таком большом количестве признаков, которые были сформированы исходными моделями глубокого обучения, довольно сложно и затратно по времени. К тому же, многие признаки, найденные моделями на изображениях, могут быть сильно скоррелированы между собой.

Будем понижать размерность исходных дескрипторов с помощью соответствующих методов. Можно уменьшить размерность входных данных до 100 или 200 признаков — этого будет достаточно, чтобы произвести кластеризацию. Попробуем подобрать необходимое количество компонент в новом пространстве признаков.

Для работы в общем масштабе признаков, воспользуемся стандартизацией и нормализацией.

In [ ]:
print("\n" + "="*60)
print("ПРЕДОБРАБОТКА ДАННЫХ: СТАНДАРТИЗАЦИЯ И УМЕНЬШЕНИЕ РАЗМЕРНОСТИ")
print("="*60)

# Импортируем необходимые библиотеки для предобработки
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Создадим словарь с дескрипторами для удобной обработки
descriptors_dict = {
    'EfficientNet': X_ef,
    'OSNet': X_osnet,
    'VDC_Color': X_color,
    'VDC_Type': X_type
}

# Словарь для хранения обработанных данных
processed_descriptors = {}

# Обрабатываем каждый тип дескрипторов
for name, X in descriptors_dict.items():
    print(f"\nОбработка {name}...")
    print(f"  Исходная размерность: {X.shape}")
    
    # 1. Стандартизация
    print("  Шаг 1: Стандартизация (StandardScaler)")
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    # 2. Уменьшение размерности до 100 компонент (PCA)
    print("  Шаг 2: Уменьшение размерности PCA до 100 компонент")
    pca = PCA(n_components=100, random_state=42)
    X_pca = pca.fit_transform(X_scaled)
    
    # Сохраняем обработанные данные
    processed_descriptors[name] = X_pca
    print(f"  Результат: {X_pca.shape}")
    print(f"  Объясненная дисперсия: {pca.explained_variance_ratio_.sum():.2%}")

print("\n" + "="*60)
print("ПРЕДОБРАБОТКА ЗАВЕРШЕНА")
print("="*60)

### Анализ результатов PCA

Мы выполнили PCA для всех четырех типов дескрипторов, уменьшив размерность до 100 компонент. Полученные результаты показывают разную степень сжатия информации:

| Модель | Исходная размерность | Объясненная дисперсия (100 компонент) |
|--------|---------------------|--------------------------------------|
| EfficientNet | 2560 | 46.15% |
| OSNet | 512 | 84.06% |
| VDC_Color | 128 | 97.07% |
| VDC_Type | 512 | 97.66% |

**Что это значит:**
- Для EfficientNet 100 компонент недостаточно (только 46% информации)
- Для VDC_Color и VDC_Type можно взять меньше компонент (информация очень сжата)
- Для OSNet 100 компонент дают хороший результат (84%)

**Следующий шаг:** подберем оптимальное количество компонент для каждого типа дескрипторов (чтобы сохранить 95% дисперсии) и выполним финальную предобработку.

In [ ]:
# Подбор оптимального количества компонент PCA и финальная предобработка

print("\n" + "="*60)
print("ПОДБОР ОПТИМАЛЬНОГО КОЛИЧЕСТВА КОМПОНЕНТ PCA")
print("="*60)

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Словарь для хранения оптимальных параметров
pca_params = {}
processed_descriptors = {}

for name, X in descriptors_dict.items():
    print(f"\nАнализ для {name} (исходная размерность: {X.shape[1]})...")
    
    # Стандартизация
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    # PCA для анализа
    pca = PCA(random_state=42)
    pca.fit(X_scaled)
    
    # Считаем кумулятивную объясненную дисперсию
    cumsum = np.cumsum(pca.explained_variance_ratio_)
    
    # Находим количество компонент для 95% дисперсии
    n_components_95 = np.argmax(cumsum >= 0.95) + 1
    variance_95 = cumsum[n_components_95 - 1] * 100
    
    print(f"  Компонент для 95% дисперсии: {n_components_95}")
    print(f"  Фактическая дисперсия: {variance_95:.2f}%")
    
    # Сохраняем параметры
    pca_params[name] = n_components_95
    
    # Выполняем финальное PCA с оптимальным количеством компонент
    print(f"  Выполняем PCA до {n_components_95} компонент...")
    pca_final = PCA(n_components=n_components_95, random_state=42)
    X_pca = pca_final.fit_transform(X_scaled)
    
    processed_descriptors[name] = X_pca
    print(f"  Результат: {X_pca.shape}")
    print(f"  Объясненная дисперсия: {pca_final.explained_variance_ratio_.sum():.2%}")

print("\n" + "="*60)
print("ПРЕДОБРАБОТКА ЗАВЕРШЕНА")
print("="*60)
print("\nИтоговые размерности:")
for name, X in processed_descriptors.items():
    print(f"  {name}: {X.shape}")

### Анализ результатов подбора оптимальной размерности

Мы выполнили PCA для каждого типа дескрипторов, подобрав количество компонент, сохраняющее 95% исходной информации.

**Итоговые размерности после предобработки:**

| Модель | Исходная размерность | Оптимальная размерность (95%) | Сжатие |
|--------|---------------------|-------------------------------|--------|
| EfficientNet | 2560 | 1807 | в 1.4 раза |
| OSNet | 512 | 235 | в 2.2 раза |
| VDC_Color | 128 | 90 | в 1.4 раза |
| VDC_Type | 512 | 46 | в 11.1 раза |

**Полученные результаты:**

1. **EfficientNet**: 2560 → 1807 признаков (95% информации)
2. **OSNet**: 512 → 235 признаков (95% информации)  
3. **VDC_Color**: 128 → 90 признаков (95% информации)
4. **VDC_Type**: 512 → 46 признаков (95% информации)

Дополнительные методы уменьшения размерности не требуются, так как задача решена — данные подготовлены для кластеризации с сохранением основного объема информации.

## 3. Моделирование и оценка качества модели

### 3.1. Кластеризация изображений

После предобработки данных приступаем к кластеризации. Для каждого типа дескрипторов применим несколько алгоритмов кластеризации и подберем оптимальное количество кластеров.

**Используемые алгоритмы:**
- **K-Means** (через MiniBatchKMeans для больших данных)
- **EM-алгоритм** (Gaussian Mixture Model)
- **Агломеративная иерархическая кластеризация** (с ограничением по глубине)
- **DBSCAN** (для поиска кластеров произвольной формы)

**Метрики качества:**
- **Calinski-Harabasz Index** (чем выше, тем лучше)
- **Davies-Bouldin Index** (чем ниже, тем лучше)

Для ускорения работы будем использовать подвыборку для некоторых алгоритмов (EM, агломеративная кластеризация), так как они требуют много памяти на полных данных.

In [ ]:
print("\n" + "="*60)
print("КЛАСТЕРИЗАЦИЯ ИЗОБРАЖЕНИЙ")
print("="*60)

from sklearn.cluster import MiniBatchKMeans
from sklearn.mixture import GaussianMixture
from sklearn.cluster import AgglomerativeClustering
from sklearn.cluster import DBSCAN
from sklearn.metrics import calinski_harabasz_score, davies_bouldin_score

# Словари для хранения результатов кластеризации
clustering_results = {}
cluster_labels = {}

# Количество кластеров для K-Means и EM (подбирается экспериментально)
# Для разных типов дескрипторов можно использовать разное количество кластеров
n_clusters = {
    'EfficientNet': 20,
    'OSNet': 15,
    'VDC_Color': 12,
    'VDC_Type': 10
}

# Применяем кластеризацию к каждому типу дескрипторов
for name, X in processed_descriptors.items():
    print(f"\n{'-'*60}")
    print(f"КЛАСТЕРИЗАЦИЯ ДЛЯ {name}")
    print(f"Размерность данных: {X.shape}")
    print(f"{'-'*60}")
    
    # Словарь для хранения результатов по данному типу дескрипторов
    results = {}
    labels_dict = {}
    
    # 1. MiniBatchKMeans (аналог K-Means для больших данных)
    print("\n1. MiniBatchKMeans...")
    k = n_clusters[name]
    kmeans = MiniBatchKMeans(n_clusters=k, batch_size=10000, random_state=42, n_init=3)
    labels_kmeans = kmeans.fit_predict(X)
    labels_dict['kmeans'] = labels_kmeans
    
    # Метрики
    ch_score = calinski_harabasz_score(X, labels_kmeans)
    db_score = davies_bouldin_score(X, labels_kmeans)
    results['kmeans'] = {
        'model': kmeans,
        'calinski_harabasz': ch_score,
        'davies_bouldin': db_score,
        'n_clusters': len(np.unique(labels_kmeans))
    }
    print(f"  Количество кластеров: {len(np.unique(labels_kmeans))}")
    print(f"  Индекс Калински-Харабаса: {ch_score:.2f}")
    print(f"  Индекс Дэвиса-Болдина: {db_score:.2f}")
    
    # 2. Gaussian Mixture Model (EM-алгоритм)
    print("\n2. Gaussian Mixture Model (EM-алгоритм)...")
    # Используем подвыборку для ускорения (10000 объектов)
    np.random.seed(42)
    idx_sample = np.random.choice(X.shape[0], size=min(20000, X.shape[0]), replace=False)
    X_sample = X[idx_sample]
    
    gmm = GaussianMixture(n_components=k, random_state=42, max_iter=100)
    labels_gmm_sample = gmm.fit_predict(X_sample)
    
    # Для полного настава предсказываем кластеры
    from scipy.spatial.distance import cdist
    
    # Предсказание для всех данных на основе близости к центрам компонент
    distances = cdist(X, gmm.means_)
    labels_gmm = np.argmin(distances, axis=1)
    labels_dict['gmm'] = labels_gmm
    
    # Метрики на полных данных
    ch_score = calinski_harabasz_score(X, labels_gmm)
    db_score = davies_bouldin_score(X, labels_gmm)
    results['gmm'] = {
        'model': gmm,
        'calinski_harabasz': ch_score,
        'davies_bouldin': db_score,
        'n_clusters': len(np.unique(labels_gmm))
    }
    print(f"  Количество кластеров: {len(np.unique(labels_gmm))}")
    print(f"  Индекс Калински-Харабаса: {ch_score:.2f}")
    print(f"  Индекс Дэвиса-Болдина: {db_score:.2f}")
    
    # 3. Агломеративная иерархическая кластеризация
    print("\n3. Agglomerative Clustering...")
    # Для иерархической кластеризации также используем подвыборку
    agg = AgglomerativeClustering(n_clusters=k)
    labels_agg_sample = agg.fit_predict(X_sample)
    
    # Для полного набора используем K-Means для назначения кластеров на основе центроидов
    # Находим центроиды кластеров на подвыборке
    centroids = np.zeros((k, X.shape[1]))
    for i in range(k):
        mask = labels_agg_sample == i
        if np.sum(mask) > 0:
            centroids[i] = np.mean(X_sample[mask], axis=0)
        else:
            centroids[i] = X_sample[np.random.randint(0, len(X_sample))]
    
    # Назначаем кластеры для всех объектов по ближайшему центроиду
    distances = cdist(X, centroids)
    labels_agg = np.argmin(distances, axis=1)
    labels_dict['agg'] = labels_agg
    
    # Метрики
    ch_score = calinski_harabasz_score(X, labels_agg)
    db_score = davies_bouldin_score(X, labels_agg)
    results['agg'] = {
        'model': agg,
        'calinski_harabasz': ch_score,
        'davies_bouldin': db_score,
        'n_clusters': len(np.unique(labels_agg))
    }
    print(f"  Количество кластеров: {len(np.unique(labels_agg))}")
    print(f"  Индекс Калински-Харабаса: {ch_score:.2f}")
    print(f"  Индекс Дэвиса-Болдина: {db_score:.2f}")
    
    # 4. DBSCAN (для поиска выбросов и кластеров произвольной формы)
    print("\n4. DBSCAN...")
    # Подбираем параметры эмпирически (eps - радиус окрестности, min_samples - мин. точек в кластере)
    # Используем подвыборку для подбора параметров
    dbscan = DBSCAN(eps=0.5, min_samples=10, n_jobs=-1)
    labels_dbscan_sample = dbscan.fit_predict(X_sample)
    
    # Количество кластеров (исключая шум)
    n_clusters_db = len(set(labels_dbscan_sample)) - (1 if -1 in labels_dbscan_sample else 0)
    n_noise = list(labels_dbscan_sample).count(-1)
    
    # Для полных данных используем тот же подход с центроидами
    unique_labels = np.unique(labels_dbscan_sample)
    valid_labels = unique_labels[unique_labels != -1]
    
    if len(valid_labels) > 0:
        # Находим центроиды для кластеров (исключая шум)
        centroids_db = np.zeros((len(valid_labels), X.shape[1]))
        for i, label in enumerate(valid_labels):
            mask = labels_dbscan_sample == label
            centroids_db[i] = np.mean(X_sample[mask], axis=0)
        
        # Назначаем кластеры для всех объектов
        distances = cdist(X, centroids_db)
        labels_dbscan = np.argmin(distances, axis=1)
        
        # Точки, которые были шумом в подвыборке, оставляем как отдельный кластер?
        # Добавим их как отдельную метку (-1)
        # Для простоты будем считать, что все точки относятся к ближайшему кластеру
    else:
        # Если кластеров не найдено, все точки - шум
        labels_dbscan = np.full(X.shape[0], -1)
    
    labels_dict['dbscan'] = labels_dbscan
    
    # Метрики (только для нешумовых точек, если их достаточно)
    mask_non_noise = labels_dbscan != -1
    if np.sum(mask_non_noise) > 100:  # Если достаточно нешумовых точек
        ch_score = calinski_harabasz_score(X[mask_non_noise], labels_dbscan[mask_non_noise])
        db_score = davies_bouldin_score(X[mask_non_noise], labels_dbscan[mask_non_noise])
    else:
        ch_score = 0
        db_score = 999
    
    results['dbscan'] = {
        'model': dbscan,
        'calinski_harabasz': ch_score,
        'davies_bouldin': db_score,
        'n_clusters': n_clusters_db,
        'n_noise': n_noise
    }
    print(f"  Количество кластеров (без шума): {n_clusters_db}")
    print(f"  Количество шумовых точек: {n_noise} ({n_noise/len(X_sample)*100:.1f}%)")
    if ch_score > 0:
        print(f"  Индекс Калински-Харабаса: {ch_score:.2f}")
        print(f"  Индекс Дэвиса-Болдина: {db_score:.2f}")
    
    # Сохраняем результаты
    clustering_results[name] = results
    cluster_labels[name] = labels_dict
    
    print(f"\nКластеризация для {name} завершена")

print("\n" + "="*60)
print("КЛАСТЕРИЗАЦИЯ ЗАВЕРШЕНА ДЛЯ ВСЕХ ТИПОВ ДЕСКРИПТОРОВ")
print("="*60)